# ImmuVis trained on Jackson, tested on Danenberg (cross-cohort virtual staining)

**What this run does:**
- Trains the **published ImmuVis code** from scratch on the Jackson train patients and tests it on the Danenberg test patients.
  - Code: Uchal et al. 2026, `anonymous.4open.science/r/ImmuVis-9173`, with `github.com/dav3794/multiplex-image-model` as a fallback.
  - Architecture: the default `train_masked_config.yaml`, i.e. **ImmuVis-Conv** (ConvNeXt marker-agnostic + pan-marker encoder, hyperkernels, heteroscedastic decoder).
- The public ImmuVis weights are not used. They were pretrained on IMC17M, which contains both cohorts.

**Channels:** the 14 channels both cohorts share (H3, DNA1, DNA2 + the 11 shared markers), each as its own marker token.
- At test time each marker is predicted from the other 13, the same inputs as every other model in the figure.

**Preprocessing (ImmuVis pipeline):**
- arcsinh(x/5) → Butterworth low-pass (cutoff 0.2) → clip at 2.3 → /2.3.
- 2.3 is the published Jackson-Basel clip limit. It is used for Danenberg too, so no Danenberg statistics are used (level A).

**Training:**
- **Loss and masking**, as published: random 128×128 crops with rotation and flips; channel-subset sampling (≥ 75%); up to 50% of channels fully masked; 60% spatial masking with 8-px patches; β-NLL loss (β = 0.5); AdamW (lr 5e-4 → 1e-5, cosine, 10% warm-up), weight decay 1e-4.
- **Changes from the published setup:**
  - fp16 autocast, because a T4 has no bf16;
  - a time budget sets the number of steps;
  - the checkpoint is chosen by leave-one-marker-out pixel r on Jackson val patients.
- **Patients:** the same Jackson train/val patients as the Jackson-trained per-cell models (replicate 0 of `train-test-grid`).

**Inference:** 128-px tiles with 32-px context, stitched. Scored exactly like `cross-cohort-metrics`.
- **Errors (RMSE, v3 of `immuvis-jackson`):** the output, clip(filtered arcsinh, 0, 2.3)/2.3, is multiplied by 2.3 and mapped to the target space with the fixed Jackson percentiles. Correlations use the raw output, as before.

**Two kernels:** `immuvis-jackson-train` trains and saves the checkpoint; `immuvis-jackson` loads it and scores Danenberg. (The first single-kernel run was killed by Kaggle after 5.5 h with no log, losing the checkpoint.)

In [ ]:
import os, glob, json, time, sys, subprocess, warnings, traceback, re, pickle, gc, random
import numpy as np, pandas as pd
import pyarrow.parquet as pq
import scipy.sparse as sp
from scipy.spatial import cKDTree
from sklearn.model_selection import GroupKFold
from sklearn.metrics import roc_auc_score
from sklearn.linear_model import LinearRegression
from joblib import Parallel, delayed
try:
    import lightgbm as lgb
except ImportError:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'lightgbm'], check=True)
    import lightgbm as lgb
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")

IN = os.environ.get("IN_ROOT", "/kaggle/input")
OUT = os.environ.get("OUT_DIR", "/kaggle/working")
SMOKE = os.environ.get("SMOKE") == "1"
SEED = 20261013
N_BOOT = 100 if SMOKE else 1000
N_BOOT_F1 = 50 if SMOKE else 200
N_SHIFT = 5
MIN_CELLS = 200
COF = 5.0
RING_R = 30.0
N_FIT = 15_000 if SMOKE else 100_000          # cells per cross-fit half (R5) and per classifier
N_FIT_FINAL = 30_000 if SMOKE else 300_000    # Jackson cells for the baselines
LGB = dict(n_estimators=40 if SMOKE else 250, learning_rate=0.06, num_leaves=31, min_child_samples=50,
           subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0, n_jobs=os.cpu_count(), verbose=-1)
ARMS = ["residual", "gamma", "immuvis"]
NUC = ["histone_h3", "dna1", "dna2"]
# Image channel orders (as in AbundanceVsStructure). Danenberg index 12 'c_erb_b_2_her2' is clone 3B5.
D_CHAN = ['histone_h3', 'sma', 'cytokeratin_5', 'cd38', 'hla_dr', 'cytokeratin_8_18', 'cd15', 'fsp1', 'cd163',
          'cd278_icos', 'cd134', 'cd68', 'c_erb_b_2_her2', 'cd3', 'podoplanin', 'cd11c', 'cd279_pd_1', 'gitr_tnfrsf18',
          'cd16', 'c_erb_b_2_her2_d8f12', 'cd45ra', 'beta_2_microglobulin', 'cd45', 'foxp3', 'cd20',
          'estrogen_receptor_alpha', 'cd8a', 'cd57', 'ki_67', 'cd140b_pdgf_receptor_beta', 'caveolin_1', 'cd4',
          'cd31_v_wf', 'cxcl12_sdf_1', 'hla_abc', 'pan_cytokeratin', 'cleaved_caspase3', 'dna1', 'dna2']
J_CHAN = ['histone_h3', 'histone_h3_trimethylate', 'cytokeratin_5', 'fibronectin', 'cytokeratin_19', 'cytokeratin_8_18',
          'twist', 'cd68', 'keratin_14_krt14', 'sma', 'vimentin', 'c_myc', 'c_erb_b_2_her2', 'cd3', 'histone_h3_phospho',
          'slug', 'estrogen_receptor_alpha', 'progesterone_receptor_a_b', 'p53', 'cd44', 'cd45', 'gata3', 'cd20',
          'carbonic_anhydrase_ix', 'e_cadherin_p_cadherin', 'ki_67', 'egfr', 's6', 'v_wf', 'm_tor', 'cytokeratin_7',
          'pan_cytokeratin', 'cleaved_parp', 'dna1', 'dna2']
D_TABLE = {"c_erb_b_2_her2": "c_erb_b_2_her2_3b5"}       # image name -> Danenberg cell-table column
MARKERS = sorted(m for m in set(D_CHAN) & set(J_CHAN) if m not in NUC)    # = task_names in the training code
assert len(MARKERS) == 11, MARKERS
STROMAL_KEYS = ("fibro", "endothel", "stroma", "pdpn", "myofib", "vessel", "vwf")
T0 = time.time()

def tick(msg):
    print(f"[{(time.time() - T0) / 60:6.1f} min] {msg}", flush=True)

def g(pattern):
    return sorted(glob.glob(f"{IN}/**/{pattern}", recursive=True))

def near(name, anchor):
    hits = g(name)
    assert hits, f"missing input {name}"
    return max(hits, key=lambda c: len(os.path.commonpath([c, anchor])))

def guard(name):
    """Run an analysis block; on failure print the traceback and carry on, so the kernel ends COMPLETE."""
    def deco(fn):
        try:
            return fn()
        except Exception:
            print(f"!!! block {name} failed:\n{traceback.format_exc()}", flush=True)
            FAILED.append(name)
            return None
    return deco
FAILED = []

## Cell tables (Danenberg cells to analyse; Jackson cells for the baselines)

In [ ]:
P = {}
for p in g("cells.parquet"):
    key = "danenberg" if "metabric_id" in pq.read_schema(p).names else "jackson"
    P[key] = dict(cells=p, contacts=near("cell_neighbours.parquet", p), report=near("prep_report.json", p))
P["danenberg"].update(clinical=near("clinical.parquet", P["danenberg"]["cells"]),
                      split=near("danenberg_patient_split.csv", P["danenberg"]["cells"]))
print(json.dumps(P, indent=1))

def load_table(key):
    cells = pd.read_parquet(P[key]["cells"])
    if key == "danenberg":
        cells = cells[(cells.is_tumour == 1) & (cells.is_hot_aggregate == 0)]
        pid = "metabric_id"
    else:
        cells = cells[cells.is_tumour == 1]
        pid = "patient_id"
    n_per = cells.groupby("image_name").size()
    cells = cells[cells.image_name.isin(n_per.index[n_per >= MIN_CELLS])]
    cells = cells.sort_values(["image_name", "object_number"]).reset_index(drop=True)
    img_code, img_names = pd.factorize(cells.image_name, sort=True)
    starts = np.r_[0, np.flatnonzero(np.diff(img_code)) + 1, len(cells)]
    loc = pd.Series(np.arange(len(cells)), index=pd.MultiIndex.from_arrays(
        [cells.image_number.values, cells.object_number.values]))
    con = pd.read_parquet(P[key]["contacts"])
    ca = loc.reindex(pd.MultiIndex.from_arrays([con.ImageNumber.values, con["from"].values])).values
    cb = loc.reindex(pd.MultiIndex.from_arrays([con.ImageNumber.values, con["to"].values])).values
    ok = ~(np.isnan(ca) | np.isnan(cb))
    ca, cb = ca[ok].astype(np.int64), cb[ok].astype(np.int64)
    ca, cb = ca[ca != cb], cb[ca != cb]
    if key == "danenberg":
        epi_frac = cells.groupby("cell_phenotype").is_epithelial.mean()
        bmap = {p: "epithelial" if epi_frac[p] > 0.5 else
                "stromal" if any(s in p.lower() for s in STROMAL_KEYS) else "immune" for p in epi_frac.index}
        broad = cells.cell_phenotype.map(bmap).astype(str).values
        epi = cells.is_epithelial.values == 1
    else:
        broad = cells.broad_class.astype(str).values
        epi = broad == "epithelial"
    return dict(key=key, cells=cells, img=img_code, img_names=np.asarray(img_names), starts=starts,
                patient=cells[pid].astype(str).values, xy=cells[["x", "y"]].values.astype(np.float64),
                ca=ca, cb=cb, broad=broad, epi=epi, phen=cells.cell_phenotype.astype(str).values,
                log_area=np.log(np.maximum(cells["area"].values, 1)).astype(np.float32))

DD, DJ = load_table("danenberg"), load_table("jackson")
N = len(DD["cells"])
split = pd.read_csv(P["danenberg"]["split"]).set_index("metabric_id").split
DD["split"] = pd.Series(DD["patient"]).map(split).fillna("none").values
clin = pd.read_parquet(P["danenberg"]["clinical"]).replace({"nan": np.nan})
clin = clin.assign(pid=clin.metabric_id.astype(str)).groupby("pid").first()
print(f"Danenberg: {N:,} cells, {len(DD['img_names'])} images, {len(set(DD['patient']))} patients; "
      f"split cells {pd.Series(DD['split']).value_counts().to_dict()}")
print(f"Jackson: {len(DJ['cells']):,} cells, {len(DJ['img_names'])} images")
pats, pcode = np.unique(DD["patient"], return_inverse=True)
W = np.random.default_rng(SEED).multinomial(len(pats), np.full(len(pats), 1 / len(pats)), size=N_BOOT).astype(np.float64)
tick("tables loaded")

def contact_matrix(D):
    n = len(D["cells"])
    C = sp.coo_matrix((np.ones(len(D["ca"]), np.float32), (D["ca"], D["cb"])), shape=(n, n)).tocsr()
    C = (C + C.T).tocsr()
    C.data[:] = 1.0
    return C

def ring_means(X, D, C):
    """Mean of X over non-touching cells within RING_R of the centroid, and their count."""
    n, p = X.shape
    Mr = np.full((n, p), np.nan, np.float32)
    nr = np.zeros(n, np.float32)
    for a, b in zip(D["starts"][:-1], D["starts"][1:]):
        m = b - a
        pr = cKDTree(D["xy"][a:b]).query_pairs(RING_R, output_type="ndarray")
        R = sp.coo_matrix((np.ones(len(pr), np.float32), (pr[:, 0], pr[:, 1])), shape=(m, m)).tocsr()
        R = (R + R.T).tocsr()
        Ci = C[a:b, a:b].tocsr()
        R = (R - R.multiply(Ci)).tocsr()
        R.eliminate_zeros()
        dr = np.asarray(R.sum(1)).ravel()
        Mr[a:b] = np.where(dr[:, None] > 0, (R @ X[a:b]) / np.maximum(dr, 1)[:, None], np.nan)
        nr[a:b] = dr
    return Mr, nr

## Danenberg test set, truth maps and scoring (shared with `cross-cohort-metrics`)

In [ ]:
if subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--only-binary=:all:", "imagecodecs==2024.9.22"]).returncode:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--only-binary=:all:", "imagecodecs"], check=True)   # 2024.9.22 no longer offered
import tifffile
import torch
dev = "cuda" if torch.cuda.is_available() else "cpu"
N_GPU = torch.cuda.device_count()

# truth cell means (target space) from kill-test, in cell-table order
KT = pd.read_parquet(g("percell_truth_pred.parquet")[0], columns=["image_name", "object_number"] + [f"truth__{m}" for m in MARKERS])
assert (KT.image_name.values == DD["cells"].image_name.values).all() and \
       (KT.object_number.values == DD["cells"].object_number.values).all(), "kill-test rows do not match the cell table"
TRUTH = {m: KT[f"truth__{m}"].values.astype(np.float32) for m in MARKERS}
del KT

tifs = g("*.tiff")
D_IMG = {os.path.splitext(os.path.basename(p))[0]: p for p in tifs
         if os.path.basename(p).startswith("MB_") and "/masks/" not in p.replace("\\", "/")}
D_MASK = {os.path.splitext(os.path.basename(p))[0]: p for p in tifs
          if os.path.basename(p).startswith("MB_") and "/masks/" in p.replace("\\", "/")}
J_IMG = {os.path.splitext(os.path.basename(p))[0]: p for p in tifs
         if not os.path.basename(p).startswith("MB_") and "/masks/" not in p.replace("\\", "/")
         and "jackson" in p.lower()}
J_MASK = {os.path.splitext(os.path.basename(p))[0]: p for p in tifs
          if not os.path.basename(p).startswith("MB_") and "/masks/" in p.replace("\\", "/")}
IMGS = list(DD["img_names"])
OBJ = DD["cells"].object_number.values.astype(np.int64)
ST = DD["starts"]
D_TE = np.flatnonzero(DD["split"] == "test")
IMG_TE = DD["img"][D_TE]
TEST_IMGS = np.unique(IMG_TE)
dix = {m: i for i, m in enumerate(D_CHAN)}
jix = {m: i for i, m in enumerate(J_CHAN)}
CACHE = os.path.dirname(g("channel_preproc_residual.pkl")[0])
_, TNS = pickle.load(open(f"{CACHE}/channel_preproc_residual.pkl", "rb"))
PIXK = pd.read_csv(g("pixel_sums_per_image.csv")[0])
PIXK = PIXK[PIXK.arm == "residual"].set_index(["marker", "image"])
print(f"Danenberg test: {len(D_TE):,} cells in {len(TEST_IMGS)} images; Jackson tiffs found: {len(J_IMG)}")

# Jackson split: replicate 0 of train-test-grid (same patients as the Jackson-trained per-cell models)
jp, jcode = np.unique(DJ["patient"], return_inverse=True)
_u = np.random.default_rng(SEED + 11 + 0).random(len(jp))
J_SPLIT = np.where(_u < 0.7, "train", np.where(_u < 0.8, "val", "test"))[jcode]
J_IMG_SPLIT = pd.Series(J_SPLIT, index=DJ["cells"].image_name.values).groupby(level=0).first()
J_TRAIN = sorted(s for s in J_IMG_SPLIT.index[J_IMG_SPLIT == "train"] if s in J_IMG)
J_VAL = sorted(s for s in J_IMG_SPLIT.index[J_IMG_SPLIT == "val"] if s in J_IMG)
print(f"Jackson images: train {len(J_TRAIN)}, val {len(J_VAL)} (tumour images of train / val patients)")

def read_hwc(path, n_chan):
    im = tifffile.imread(path)
    if im.ndim == 3 and im.shape[0] == n_chan and im.shape[-1] != n_chan:
        im = np.moveaxis(im, 0, -1)
    assert im.shape[-1] == n_chan, (path, im.shape)
    return im.astype(np.float32)

def truth_map(im_hwc, m):
    """The image models' target space: clip(arcsinh(x/5) / p99_Jackson, 0, 1) (p01 = 0)."""
    p01, p99 = TNS[jix[m]]
    return np.clip((np.arcsinh(im_hwc[..., dix[m]] / COF) - p01) / (p99 - p01 + 1e-8), 0., 1.)

def to_target_arc(m, arc):
    """arcsinh(x/5) values -> the target space (fixed Jackson-train p01/p99 of marker m)."""
    p01, p99 = TNS[jix[m]]
    return np.clip((arc - p01) / (p99 - p01 + 1e-8), 0., 1.)

def _ref_one(img_path, mask_path, n_chan, idx):
    im = read_hwc(img_path, n_chan)
    lab = tifffile.imread(mask_path).astype(np.int64).ravel()
    assert lab.size == im.shape[0] * im.shape[1], (img_path, im.shape)
    v = lab > 0
    cnt = np.bincount(lab[v])
    ok = cnt > 0
    out = {}
    for m in MARKERS:
        arc = np.arcsinh(im[..., idx[m]] / COF)
        lo, hi = np.percentile(arc, [0.5, 99.5])
        t = to_target_arc(m, arc).ravel().astype(np.float64)
        cm = np.bincount(lab[v], t[v])[ok] / cnt[ok]
        out[m] = (lo, hi, t.sum(), t.size, cm.sum(), ok.sum())
    return out

def source_reference(names, img_map, mask_map, n_chan, idx, cohort):
    """Training-cohort statistics, per marker, that need no test data:
    eva_lo / eva_hi: median over images of the per-image arcsinh p0.5 / p99.5 (maps EVA's per-image scale back);
    pixel_mean / cell_mean: mean target-space value over all pixels / over all cells (the constant-prediction baseline)."""
    res = Parallel(n_jobs=4)(delayed(_ref_one)(img_map[s], mask_map[s], n_chan, idx) for s in names)
    rows = []
    for m in MARKERS:
        a = np.array([r[m] for r in res], np.float64)
        rows.append(dict(cohort=cohort, marker=m, n_images=len(res), eva_lo=float(np.median(a[:, 0])),
                         eva_hi=float(np.median(a[:, 1])), pixel_mean=a[:, 2].sum() / a[:, 3].sum(),
                         cell_mean=a[:, 4].sum() / a[:, 5].sum(), n_cells=int(a[:, 5].sum())))
    return pd.DataFrame(rows).set_index("marker")

def rmse_from(n, Sgg, Spp, Sgp):
    return np.sqrt(np.maximum(Sgg - 2 * Sgp + Spp, 0) / n)

def cell_index(i):
    """Pixel -> row in the test arrays, for test image i."""
    m = tifffile.imread(D_MASK[IMGS[i]]).astype(np.int64)
    a, b = ST[i], ST[i + 1]
    lut = np.full(max(int(m.max()), int(OBJ[a:b].max())) + 1, -1, np.int64)
    lut[OBJ[a:b]] = np.searchsorted(D_TE, np.arange(a, b))
    return lut[m]

def r_from(n, Sg, Sp, Sgg, Spp, Sgp):
    with np.errstate(invalid="ignore", divide="ignore"):
        return (Sgp - Sg * Sp / n) / np.sqrt((Sgg - Sg ** 2 / n) * (Spp - Sp ** 2 / n))

def r4_point(a, b):
    ok = np.isfinite(a) & np.isfinite(b)
    a, b, im = a[ok].astype(np.float64), b[ok].astype(np.float64), IMG_TE[ok]
    k = int(im.max()) + 1
    cnt = np.maximum(np.bincount(im, minlength=k), 1)
    a = a - (np.bincount(im, a, k) / cnt)[im]
    b = b - (np.bincount(im, b, k) / cnt)[im]
    return float((a * b).sum() / np.sqrt((a * a).sum() * (b * b).sum()))

PATCH = 32                                        # 32 µm patches (1 µm/px), non-overlapping, partial edge tiles dropped

def patch_means(A):
    h, w = (A.shape[0] // PATCH) * PATCH, (A.shape[1] // PATCH) * PATCH
    return A[:h, :w].reshape(h // PATCH, PATCH, w // PATCH, PATCH).mean((1, 3)).ravel()

def patch_row(MODEL, m, s, G2, P2, T2=None):
    """Per-image sums over 32 µm patch means (truth G2, raw prediction P2, target-scale prediction T2)."""
    g, p = patch_means(G2), patch_means(P2)
    row = dict(model=MODEL, marker=m, image=s, n=g.size, Sg=g.sum(), Sgg=g @ g, Sp=p.sum(), Spp=p @ p, Sgp=g @ p)
    if T2 is not None:
        t = patch_means(T2)
        row.update(Sp_t=t.sum(), Spp_t=t @ t, Sgp_t=g @ t)
    if len(PATCH_CHECK) < 4 and g.size > 2 and g.std() > 0 and p.std() > 0:   # sums must give the direct patch r
        PATCH_CHECK.append(abs(float(np.corrcoef(g, p)[0, 1]) - float(r_from(*(row[c] for c in ["n", "Sg", "Sp", "Sgg", "Spp", "Sgp"])))))
    return row

PATCH_CHECK = []

def score_model(MODEL, predict_image, to_target=None):
    """predict_image(im_hwc) -> {marker: prediction map (H, W)}. Writes pixel sums and per-cell predictions.
    to_target(m, P) maps the model's output scale to the target space without test data; its pixel sums (Sp_t, Spp_t,
    Sgp_t) and per-cell means (predt__) give the errors (RMSE). Correlations use the raw output, as before."""
    rows, prow = [], []
    CELLP = {m: np.full(len(D_TE), np.nan, np.float32) for m in MARKERS}
    CELLT = {m: np.full(len(D_TE), np.nan, np.float32) for m in MARKERS}
    for j, i in enumerate(TEST_IMGS):
        s = IMGS[i]
        im = read_hwc(D_IMG[s], len(D_CHAN))
        preds = predict_image(im)
        loc = cell_index(i)
        v = loc >= 0
        for m in MARKERS:
            G = truth_map(im, m).astype(np.float64).ravel()
            Pm = preds[m].astype(np.float64)
            assert Pm.shape == im.shape[:2], (s, m, Pm.shape)
            Pf = Pm.ravel()
            row = dict(model=MODEL, marker=m, image=s, n=G.size, Sg=G.sum(), Sgg=G @ G, Sp=Pf.sum(), Spp=Pf @ Pf, Sgp=G @ Pf)
            if to_target is not None:
                Tm = to_target(m, Pm).astype(np.float64)
                Tf = Tm.ravel()
                row.update(Sp_t=Tf.sum(), Spp_t=Tf @ Tf, Sgp_t=G @ Tf)
            if j < 3:   # the truth map must equal kill-test's
                k_ = PIXK.loc[(m, s)]
                assert abs(row["Sg"] - k_.Sg) <= 1e-4 * max(1.0, abs(k_.Sg)) and row["n"] == k_.n, (s, m, row["Sg"], k_.Sg)
            rows.append(row)
            prow.append(patch_row(MODEL, m, s, G.reshape(Pm.shape), Pm, Tm if to_target is not None else None))
            cid = loc[v]
            cnt = np.bincount(cid, minlength=len(D_TE))
            sm = np.bincount(cid, Pm[v], len(D_TE))
            got = cnt > 0
            CELLP[m][got] = (sm[got] / cnt[got]).astype(np.float32)
            if to_target is not None:
                CELLT[m][got] = (np.bincount(cid, Tm[v], len(D_TE))[got] / cnt[got]).astype(np.float32)
        if j % 20 == 0:
            tick(f"{MODEL}: scored {j + 1}/{len(TEST_IMGS)} test images")
    S = pd.DataFrame(rows)
    tag = MODEL.lower().replace(" ", "-")
    S.to_csv(f"{OUT}/pixel_sums_test_{tag}.csv", index=False)
    pd.DataFrame(prow).to_csv(f"{OUT}/patch_sums_test_{tag}.csv", index=False)
    assert PATCH_CHECK and max(PATCH_CHECK) < 1e-5, PATCH_CHECK
    np.savez_compressed(f"{OUT}/percell_pred_{tag}.npz", image=np.asarray(IMGS)[IMG_TE], object_number=OBJ[D_TE],
                        **{f"pred__{MODEL}__{m}": CELLP[m] for m in MARKERS},
                        **({f"predt__{MODEL}__{m}": CELLT[m] for m in MARKERS} if to_target is not None else {}))
    # point summaries (CIs are computed locally with the other models, same bootstrap draws)
    out = []
    for m in MARKERS:
        d = S[S.marker == m]
        ri = r_from(d.n.values, d.Sg.values, d.Sp.values, d.Sgg.values, d.Spp.values, d.Sgp.values)
        row = dict(marker=m, pooled=float(r_from(*(d[c].sum() for c in ["n", "Sg", "Sp", "Sgg", "Spp", "Sgp"]))),
                   per_image=float(np.nanmean(ri)), R4=r4_point(TRUTH[m][D_TE], CELLP[m]))
        if to_target is not None:
            row.update(pixel_rmse=float(rmse_from(d.n.sum(), d.Sgg.sum(), d.Spp_t.sum(), d.Sgp_t.sum())),
                       per_image_rmse=float(np.mean(rmse_from(d.n.values, d.Sgg.values, d.Spp_t.values, d.Sgp_t.values))),
                       cell_rmse=float(np.sqrt(np.nanmean((CELLT[m] - TRUTH[m][D_TE]) ** 2))))
        out.append(row)
    out = pd.DataFrame(out)
    out.to_csv(f"{OUT}/metrics_point_{tag}.csv", index=False)
    print(out.round(3).to_string(index=False))
    print("mean over 10 markers (ERα excluded):",
          out[out.marker != "estrogen_receptor_alpha"].drop(columns="marker").mean().round(3).to_dict())
    return S, CELLP

## ImmuVis code and model configuration

In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "comet_ml", "ruamel.yaml"], check=True)
import zipfile, urllib.request, copy
from torch.amp import autocast, GradScaler
import torchvision.transforms.functional as TF
from skimage.filters import butterworth

CODE = "/tmp/immuvis_code"
def fetch_code():
    try:
        urllib.request.urlretrieve("https://anonymous.4open.science/api/repo/ImmuVis-9173/zip", "/tmp/immuvis.zip")
        zipfile.ZipFile("/tmp/immuvis.zip").extractall(CODE)
        return "anonymous.4open.science/r/ImmuVis-9173"
    except Exception as e:
        print("anonymous release not available:", e)
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/dav3794/multiplex-image-model.git", CODE], check=True)
        return "github.com/dav3794/multiplex-image-model"
CODE_SOURCE = fetch_code()
root = CODE if os.path.isdir(f"{CODE}/multiplex_model") else os.path.dirname(glob.glob(f"{CODE}/**/multiplex_model", recursive=True)[0])
sys.path.insert(0, root)
from multiplex_model.modules import MultiplexAutoencoder
from multiplex_model.utils import (EncoderConfig, DecoderConfig, apply_channel_masking, apply_spatial_masking,
                                   ClampWithGrad, get_scheduler_with_warmup)
from multiplex_model.losses import beta_nll_loss
print("ImmuVis code from", CODE_SOURCE)
# cuDNN on a T4 ran this model at ~6 s/step and leaked ~50 MB host RAM per step (immuvis-profile);
# with cuDNN off: 0.58 s/step and flat RAM
torch.backends.cudnn.enabled = False

CH = NUC + MARKERS                         # 14 shared channels = 14 marker tokens
IDS = torch.arange(len(CH))
CLIP = 2.3
CROP, BATCH = 128, 16
TRAIN_MIN = float(os.environ.get("TRAIN_MIN", 180))
EVAL_EVERY = 500

def immuvis_pre(im_hwc, names, idx):
    x = np.arcsinh(np.stack([im_hwc[..., idx[c]] for c in names]) / COF)
    x = np.stack([butterworth(c, cutoff_frequency_ratio=0.2, high_pass=False) for c in x])
    return (np.clip(x, 0, CLIP) / CLIP).astype(np.float16)

ENC = EncoderConfig(ma_layers_blocks=[4], ma_embedding_dims=[16], pm_layers_blocks=[4, 4, 4],
                    pm_embedding_dims=[192, 384, 768],
                    hyperkernel=dict(kernel_size=1, padding=0, stride=1, use_bias=True)).model_dump()
DEC = DecoderConfig(decoded_embed_dim=512, num_blocks=1,
                    hyperkernel=dict(kernel_size=1, padding=0, stride=1, use_bias=True)).model_dump()

## Load the checkpoint from `immuvis-jackson-train`; inference on the Danenberg test patients and scoring

In [ ]:
CK = torch.load(g("immuvis_jackson_best.pth")[0], map_location="cpu", weights_only=False)
assert CK["channels"] == CH and CK["clip"] == CLIP
net = MultiplexAutoencoder(num_channels=len(CH), encoder_config=ENC, decoder_config=DEC).to(dev)
net.load_state_dict(CK["model_state_dict"])
net.eval()
best = CK["best_val_loo_pixel_r"]
print(f"checkpoint: step {CK['step']}/{CK['total_steps']}, Jackson val LOO pixel r {best:.3f}")

T_IN, M_CTX = 128, 32

@torch.no_grad()
def immuvis_predict(im_hwc):
    x = immuvis_pre(im_hwc, CH, dix)               # [14, H, W], Danenberg channel order mapped by name
    C, H, W = x.shape
    tiles = [(y0, x0) for y0 in range(0, H, T_IN) for x0 in range(0, W, T_IN)]
    win = T_IN + 2 * M_CTX
    xp = np.pad(x, ((0, 0), (M_CTX, M_CTX + T_IN), (M_CTX, M_CTX + T_IN)))
    batch = torch.from_numpy(np.stack([xp[:, y0:y0 + win, x0:x0 + win] for y0, x0 in tiles]).astype(np.float32))
    preds = {}
    for k in range(3, len(CH)):
        act = [i for i in range(len(CH)) if i != k]
        outs = []
        for b0 in range(0, len(batch), 64):
            xb = batch[b0:b0 + 64].to(dev)
            with autocast(device_type="cuda", dtype=torch.float16, enabled=dev == "cuda"):
                lat = net.encode(xb[:, act], IDS[act].to(dev).expand(len(xb), -1))["output"]
                o = net.decode(lat, IDS[[k]].to(dev).expand(len(xb), -1))
            outs.append(torch.sigmoid(o[..., 0].float())[:, 0, M_CTX:M_CTX + T_IN, M_CTX:M_CTX + T_IN].cpu().numpy())
        outs = np.concatenate(outs)
        P = np.zeros((H + T_IN, W + T_IN), np.float32)
        for (y0, x0), o in zip(tiles, outs):
            P[y0:y0 + T_IN, x0:x0 + T_IN] = o
        preds[CH[k]] = P[:H, :W]
    return preds

try:                                              # the checkpoint is already saved; a scoring bug must not lose it
    # ImmuVis output is clip(filtered arcsinh, 0, 2.3) / 2.3, so CLIP * output is back in arcsinh units
    S_IV, CELL_IV = score_model("ImmuVis", immuvis_predict, lambda m, P: to_target_arc(m, CLIP * P))
except Exception:
    print("!!! scoring failed:", traceback.format_exc(), flush=True)
    FAILED.append("scoring")
json.dump({"code_source": CODE_SOURCE, "train_minutes": CK["train_minutes"], "total_steps": CK["total_steps"],
           "best_step": CK["step"], "best_val_loo_pixel_r": best, "minutes": (time.time() - T0) / 60,
           "n_jackson_train_images": CK["n_jackson_train_images"], "n_jackson_val_images": CK["n_jackson_val_images"],
           "failed_blocks": FAILED},
          open(f"{OUT}/run_info.json", "w"))
tick("done")